# Module 4: Multi-Agentic Patterns — Centralized Supervisors, Decentralized Handoffs, & Sub-Graphs

### Objective
Master the architectural layout of complex multi-agent systems. You will learn how to coordinate multiple LLMs by balancing centralized orchestrators against decentralized agent-to-agent handoffs, and safely isolate execution using sub-graph boundaries to avoid state pollution.

In [ ]:
# Install the core LangGraph suite, consolidated Google GenAI integrations, and authentication tools
!pip install -U langgraph langchain-google-genai google-cloud-aiplatform pydantic

In [ ]:
import os
import sys

# 1. Authenticate to Google Cloud (skip if running inside a pre-authenticated Vertex AI Workbench)
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab user authenticated successfully.")

# 2. Configure your GCP Project variables
GCP_PROJECT = input("Enter your Google Cloud Project ID: ").strip()
GCP_LOCATION = input("Enter your GCP Region (e.g., us-central1): ").strip() or "us-central1"

# Point the modern ChatGoogleGenerativeAI class to your enterprise Vertex AI backend
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
os.environ["GOOGLE_CLOUD_PROJECT"] = GCP_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION

# Initialize the Vertex AI SDK for environmental tools
from google.cloud import aiplatform
aiplatform.init(project=GCP_PROJECT, location=GCP_LOCATION)
print(f"Vertex AI routing successfully initialized using the unified Google GenAI SDK.")

## Conceptual Multi-Agent Architectures

Before writing code, it is vital to visualize how data and control move across agent boundaries. In this notebook, we evaluate three design patterns:

1. **Centralized (Supervisor):** A single "Hub" node handles the user request, decides which specialized "Spoke" agent should run next, receives their output, and re-evaluates the state. Spokes *never* talk directly to each other.
2. **Decentralized (Handoff Swarm):** There is no coordinator. Agents directly transfer control to one another using LangGraph's native `Command(goto=...)` primitives from within their execution logic.
3. **Isolated State Sub-Graphs:** Specialized workflows are wrapped in their own sandboxed graphs containing a private state. The parent graph feeds input into the child, and maps only the finalized product back up to the master state, protecting keys from being cross-contaminated or overwritten.

### 🛠️ Exercise: The Supervisor Pattern (Centralized Orchestration)

#### Goal
Build a centralized routing Research Team consisting of a Supervisor node, a WebScraper agent node, and a Writer agent node.

#### Key Concepts to Observe
* **Centralized Routing**: The supervisor inspects the shared graph history and decides the next step via structured routing flags (`"WebScraper" | "Writer" | "FINISH"`).
* **State Consolidation**: Specialized worker agents execute their targeted tasks and automatically pass control directly back to the supervisor node upon completion.

#### Instructions
1. Define a state dictionary/Pydantic model containing a `messages` list and a `next` destination routing field string.
2. Implement the `supervisor` function using a structured model output or tool parser to select the next step.
3. Use `graph.add_node()` and connect them using `graph.add_conditional_edges()` running out from your supervisor hub.
4. Run the code block below to test your implementation against the verification assertion checks.

In [ ]:
from typing import Annotated, Dict, Any, List, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the production-flagship model
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=.1, location="us")

# 1. State definition
class TeamState(TypedDict):
    messages: Annotated[list, add_messages]
    next: str

# Define the Pydantic schema for structured supervisor decisions
class RouteResponse(BaseModel):
    next_step: Literal["WebScraper", "Writer", "FINISH"] = Field(
        description="Select the best next worker node based on history, or FINISH if the task is fully complete."
    )

# 2. Implement the Supervisor Node
def supervisor_node(state: TeamState) -> Dict[str, Any]:
    print("--- HUB: Supervisor Decision Loop ---")
    structured_llm = llm.with_structured_output(RouteResponse)

    # System prompt directing the supervisor to manage the data pipeline
    system_prompt = (
        "You are the manager of a research team. Your workers are 'WebScraper' (gathers data) "
        "and 'Writer' (synthesizes findings). Analyze the message trail and assign the next worker. "
        "If the final report is complete and polished, respond with FINISH."
    )

    messages_payload = [HumanMessage(content=system_prompt)] + state["messages"]
    response = structured_llm.invoke(messages_payload)

    print(f"[Supervisor Decision]: Routing task to -> {response.next_step}")
    return {"next": response.next_step}

# 3. Implement Worker Nodes
def web_scraper_node(state: TeamState) -> Dict[str, Any]:
    print("--- SPOKE: Executing WebScraper Agent ---")
    # Simulating data extraction tool-run
    return {"messages": [AIMessage(content="[WebScraper Output]: Gathered server latency metrics and CPU cloud profile logs.")]}

def writer_node(state: TeamState) -> Dict[str, Any]:
    print("--- SPOKE: Executing Writer Agent ---")
    # Simulating synthesis report generation
    return {"messages": [AIMessage(content="[Writer Output]: Final Executive Summary: Systems operating normally with 99.9% uptime.")]}

# 4. Assemble Graph Topology
research_builder = StateGraph(TeamState)

research_builder.add_node("supervisor", supervisor_node)
research_builder.add_node("WebScraper", web_scraper_node)
research_builder.add_node("Writer", writer_node)

# Workers always report back to the supervisor hub
research_builder.add_edge("WebScraper", "supervisor")
research_builder.add_edge("Writer", "supervisor")

# The central hub conditionally determines routing paths
research_builder.add_conditional_edges(
    "supervisor",
    lambda state: state["next"],
    {
        "WebScraper": "WebScraper",
        "Writer": "Writer",
        "FINISH": END
    }
)

research_builder.add_edge(START, "supervisor")
supervisor_graph = research_builder.compile()

In [ ]:
# Verify full structural orchestration
print("--- Testing Supervisor Pipeline ---")
test_input = {"messages": [HumanMessage(content="Scrape infrastructure logs and compile a brief summary report.")], "next": ""}
execution_flow = supervisor_graph.invoke(test_input)

print("\n--- Asserting State Invariants ---")
assert "messages" in execution_flow, "Pipeline state missing consolidated message array."
assert len(execution_flow["messages"]) >= 3, "Pipeline didn't successfully transit through all workers."
print("Assertion Passed! Supervisor graph executed and completed all loops successfully.")

In [ ]:
print("\n--- Testing Supervisor Pipeline (Directly to Writer) ---")
writer_test_input = {"messages": [HumanMessage(content="The data has been gathered, please compile the summary report.")], "next": ""}
writer_execution_flow = supervisor_graph.invoke(writer_test_input)

print("\n--- Asserting State Invariants for Writer Direct Call ---")
assert "messages" in writer_execution_flow, "Pipeline state missing consolidated message array."
# We expect the initial human message, the supervisor's decision to route to writer, and the writer's output.
assert len(writer_execution_flow["messages"]) >= 2, "Pipeline didn't successfully transit through writer."
assert "Writer Output" in writer_execution_flow["messages"][-1].content, "Writer agent was not called or did not produce expected output."
print("Assertion Passed! Supervisor graph directly called Writer and completed successfully.")

### 🛠️ Exercise: Decentralized Handoffs (Edgeless Architecture)

#### Goal
Create a customer support routing graph where individual agents transfer execution control dynamically using the `Command` object, removing the need for manual conditional edges in the blueprint.

#### Key Concepts to Observe
* **Edgeless Control Flow**: Nodes return a `Command(goto="NodeName", update={...})` object. This merges state changes and topological path execution directly within your code runtime.
* **Decentralized Autonomy**: Control shifts instantly without moving through a central hub node, simplifying graph code maintenance.

#### Instructions
1. Define a state dictionary/Pydantic model containing `messages`.
2. Implement the `initial_triage` function. Evaluate user intent and return a `Command` object targeting the respective downstream agent.
3. Use `graph.add_node()` to list the nodes, completely omitting conditional routing logic from the graph architecture initialization block.
4. Run the code block below to test your implementation against the verification assertion checks.

In [ ]:
from langgraph.types import Command

# 1. State Schema definition
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]

# 2. Implement the Initial Triage Node with Dynamic Command Handoffs
def initial_triage(state: SupportState):
    print("--- NODE: Initial Triage ---")
    user_issue = state["messages"][-1].content.lower()

    # Evaluate inputs inline and return explicit runtime routing commands
    if "billing" in user_issue or "invoice" in user_issue:
        print("[Triage Redirect]: Directing to Billing Department.")
        return Command(
            goto="BillingAgent",
            update={"messages": [AIMessage(content="Routing your request to our accounts team.")]}
        )
    else:
        print("[Triage Redirect]: Directing to Tech Department.")
        return Command(
            goto="TechAgent",
            update={"messages": [AIMessage(content="Routing your request to senior systems support engineers.")]}
        )

# 3. Implement Department Specialists
def tech_agent(state: SupportState) -> Dict[str, Any]:
    print("--- NODE: Tech Agent Spec ---")
    return {"messages": [AIMessage(content="[Tech Fixed]: Reset system flags. SSH access restored.")]}

def billing_agent(state: SupportState) -> Dict[str, Any]:
    print("--- NODE: Billing Agent Spec ---")
    return {"messages": [AIMessage(content="[Billing Fixed]: Credited overcharged balance back to your subscription profile.")]}

# 4. Compile the Edgeless Architecture Layout
support_builder = StateGraph(SupportState)

support_builder.add_node("initial_triage", initial_triage)
support_builder.add_node("TechAgent", tech_agent)
support_builder.add_node("BillingAgent", billing_agent)

support_builder.add_edge(START, "initial_triage")
# Notice: No custom conditional edge logic or normal edges connect initial_triage to the agents!
support_builder.add_edge("TechAgent", END)
support_builder.add_edge("BillingAgent", END)

support_graph = support_builder.compile()

# --- Execution Verification Run ---
print("\n--- Testing Route Path A (Billing Context) ---")
billing_message = "Can you verify my last invoice billing issue?"
print(f"Human Message: {billing_message}")
billing_flow = support_graph.invoke({"messages": [HumanMessage(content=billing_message)]})
print(f"Final Agent Output: {billing_flow['messages'][-1].content}\n")

print("--- Testing Route Path B (Technical Context) ---")
tech_message = "My remote database cluster is throwing timeout connections."
print(f"Human Message: {tech_message}")
tech_flow = support_graph.invoke({"messages": [HumanMessage(content=tech_message)]})
print(f"Final Agent Output: {tech_flow['messages'][-1].content}")

assert "accounts team" in billing_flow["messages"][1].content, "Failed dynamic routing to Billing target."
assert "systems support" in tech_flow["messages"][1].content, "Failed dynamic routing to Tech target."
print("\nAssertion Passed! Handoff architecture executed cleanly with zero formal conditional edge structures.")

### 🛠️ Exercise: Isolated State Sub-Graphs

#### Goal
Build a parent-child graph layout with a private `ChildState` to isolate scratchpad variables and eliminate state pollution in the master pipeline.

#### Key Concepts to Observe
* **State Pollution Mitigation**: Scratchpad logs, raw text extracts, and intermediate tokens should remain localized inside the sub-graph worker.
* **State Bridge Mapping**: The parent node calls the child graph sub-pipeline explicitly, parsing specific fields inward and extracting only specified outputs to merge clean results back up to the master state.

#### Instructions
1. Define a state dictionary/Pydantic model containing `final_report` (ParentState) and another dictionary for `scratchpad` and `raw_data` inputs (ChildState).
2. Implement the internal child sub-graph workflow node functions.
3. Use `graph.add_node()` inside your master graph definition, mapping the child compiled graph execution output directly into the primary pipeline.
4. Run the code block below to test your implementation against the verification assertion checks.

In [ ]:
# 1. Define the Private Child Sub-Graph State
class ChildState(TypedDict):
    raw_document: str
    regex_scratchpad: str
    extracted_metrics: List[str]

# Implement Child Nodes (Isolated Sub-Graph Operations)
def extract_raw_text(state: ChildState) -> Dict[str, Any]:
    print(" -> [Sub-Graph Node]: Extracting raw payload tokens")
    return {"regex_scratchpad": f"Processing metrics string from chunk: {state['raw_document']}"}

def clean_and_parse(state: ChildState) -> Dict[str, Any]:
    print(" -> [Sub-Graph Node]: Building analytics summary metrics")
    return {"extracted_metrics": ["CPU: 12%", "Memory: 84%", "IOPS: 4500"]}

# Compile the Child Sub-Graph
child_builder = StateGraph(ChildState)
child_builder.add_node("extract_raw_text", extract_raw_text)
child_builder.add_node("clean_and_parse", clean_and_parse)
child_builder.add_edge(START, "extract_raw_text")
child_builder.add_edge("extract_raw_text", "clean_and_parse")
child_builder.add_edge("clean_and_parse", END)
compiled_subgraph = child_builder.compile()


# 2. Define the Master Parent State
class ParentState(TypedDict):
    master_document: str
    final_report: str

# Implement Parent Master Nodes
def initial_loader(state: ParentState) -> Dict[str, Any]:
    print("--- PARENT NODE: Loading Master System Strings ---")
    return {"master_document": "System diagnostics dump log string: ID_9824X"}

# This orchestrator node bridges the gap between parent and child states
def call_subgraph_orchestrator(state: ParentState) -> Dict[str, Any]:
    print("--- PARENT NODE: Invoking Isolated Sub-Graph Pipeline ---")

    # Map fields inward into the child graph state schema
    child_inputs = {"raw_document": state["master_document"]}
    subgraph_output = compiled_subgraph.invoke(child_inputs)

    # Extract only the targeted data metrics to keep ParentState free from scratchpad pollution
    metrics_summary = ", ".join(subgraph_output["extracted_metrics"])
    return {"final_report": f"Parent Document Summary Report: [{metrics_summary}]"}

# 3. Assemble Master Parent Topology
parent_builder = StateGraph(ParentState)
parent_builder.add_node("initial_loader", initial_loader)
parent_builder.add_node("call_subgraph_orchestrator", call_subgraph_orchestrator)

parent_builder.add_edge(START, "initial_loader")
parent_builder.add_edge("initial_loader", "call_subgraph_orchestrator")
parent_builder.add_edge("call_subgraph_orchestrator", END)

master_graph = parent_builder.compile()

In [ ]:
# Execute master graph context
print("--- Starting Master Execution Stream ---")
master_results = master_graph.invoke({"master_document": "", "final_report": ""})

print("\n--- Verifying Isolated State Integrity ---")
print(f"Final Report Value:\n{master_results['final_report']}")

# Ensure state keys from ChildState did not pollute or cross-contaminate the ParentState keys
assert "final_report" in master_results, "Missing finalized report inside master context tracking."
assert "regex_scratchpad" not in master_results, "CRITICAL ERROR: State Pollution Detected! Child scratchpad keys found in parent context."
assert "extracted_metrics" not in master_results, "CRITICAL ERROR: State Pollution Detected! Child metric keys found in parent context."

print("\nAssertion Passed! Sub-graph memory parameters remained completely sandboxed.")